# Chapter 61: Complete Case Study — Building NetOps AI

**Volume 3 — Advanced Techniques for Production**

**The BGP incident at 3AM.** 12 routers lose their BGP sessions simultaneously.
Without AI: 3 hours of manual log correlation to find a one-line fix.
With NetOps AI: **56 seconds from detection to resolution.**

This is the synthesis chapter. Every technique from the book assembled into one working system.

### What You Will Build — 5 Layers of the AI Fabric

| Layer | Demo | What It Does |
|-------|------|-------------|
| **1. Data Integration** | Event Stream + Windowing | Ingest syslog, BGP events, SNMP; correlate into incidents |
| **2. Knowledge Graph** | Topology + F-score RCA | Graph traversal finds root cause with 89% confidence |
| **3. Semantic Layer** | RAG Runbook Retrieval | Retrieve relevant docs to enrich AI context |
| **4. Multi-Agent Triage** | Supervisor + Specialists | Orchestrated investigation with HITL approval |
| **5. Full Incident Lifecycle** | End-to-End 56-Second Fix | Alert to resolution, everything wired together |

### The Core Insight
> **AI does not replace network engineers. It eliminates the manual correlation work —
> the grep-through-syslog, the compare-configs, the find-the-change-in-ServiceNow.
> The engineer's judgment applies at the decision point, not the data collection point.**

In [ ]:
# pip install anthropic
# Complete NetOps AI platform — pure Python, no heavy frameworks!

import os, json, time, math, random, re, hashlib, threading
from collections import deque, defaultdict
from dataclasses import dataclass, field
from typing import Optional

# ── Anthropic client ───────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("Anthropic API connected")
else:
    client = None
    print("No API key - simulation mode (full platform still runs!)")

MODEL = "claude-haiku-4-5-20251001"

CALL_COUNT = [0]

def llm(prompt: str, system: str = "", max_tokens: int = 300) -> str:
    CALL_COUNT[0] += 1
    if USE_API:
        kwargs = {"model": MODEL, "max_tokens": max_tokens,
                  "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    return _sim(prompt, system)

def _sim(prompt: str, system: str = "") -> str:
    p = (prompt + system).lower()
    if "plan" in p or "investigat" in p or "supervisor" in p:
        return json.dumps({"investigations": ["bgp_diagnosis", "config_analysis", "topology_impact"]})
    elif "root cause" in p or "diagnosis" in p or "mtu" in p:
        return json.dumps({
            "finding": "MTU mismatch on provider-edge-02 ge-0/0/3 (changed 9000->1500 at 03:12:47)",
            "confidence": 0.89,
            "evidence": ["NOTIFICATION code 6 on 12 sessions", "config diff commit a3f9b2c", "all sessions share PE-02 neighbor"]
        })
    elif "config" in p or "diff" in p or "change" in p:
        return json.dumps({
            "finding": "MTU changed from 9000 to 1500 on ge-0/0/3 by automated provisioning script at 03:12:47",
            "confidence": 0.95,
            "evidence": ["commit a3f9b2c", "change-ticket-11847", "ge-0/0/3 MTU 1500"]
        })
    elif "performance" in p or "metric" in p or "traffic" in p:
        return json.dumps({
            "finding": "Traffic black-holed on 3 ECMP paths since 03:14:22 - 40% packet loss",
            "confidence": 0.82,
            "evidence": ["NetFlow shows 40% drop in bytes on affected paths", "SNMP counters rising"]
        })
    elif "synthesize" in p or "report" in p or "summary" in p:
        return ("INCIDENT REPORT\n"
                "Root Cause: MTU mismatch on provider-edge-02 ge-0/0/3 (9000->1500, ticket-11847)\n"
                "Impact: 12 BGP sessions, 40% traffic loss on 3 ECMP paths\n"
                "Confidence: 89%\n"
                "Proposed Fix: revert MTU to 9000 on provider-edge-02 ge-0/0/3\n"
                "Risk: LOW (reversible, narrow blast radius, known pattern)\n"
                "Recommended Action: APPROVE for autonomous execution")
    elif "runbook" in p or "retrieve" in p or "relevant" in p:
        return ("BGP session instability runbook: check MTU on transit links, "
                "verify hold-timer configuration, compare BGP timer settings between peers.")
    else:
        return f"Network AI analysis for: {prompt[:60]}..."

print(f"Model: {MODEL}")
print("NetOps AI Platform loading all 5 layers...")

## Layer 1: Data Integration — Event Stream + Windowing

Every piece of telemetry flows into a **unified event stream** (Kafka in production).
Three critical functions happen here:

1. **Event correlation**: group related events from different devices into one incident
2. **Session windowing**: close a session after N seconds of quiet (OSPF-style silence)
3. **Stream enrichment**: join raw events with CMDB to add device role, criticality, services

> **In Simple Words:** This layer is your NetFlow + syslog + SNMP collector.
> The difference: instead of just storing events, the AI layer groups them into incidents
> using session windows — exactly like OSPF uses dead timers to close an adjacency session.
>
> A session window closes after 10 seconds of no new related events.
> That session becomes one incident record — one ticket, one AI triage.

**Without this layer:** 500 individual syslog alerts in the NOC dashboard (noise).
**With this layer:** 1 incident record with all 500 events correlated (signal).

In [ ]:
# ── Layer 1: Event Stream + Windowing + Enrichment ────────────────────────────

@dataclass
class TelemetryEvent:
    event_id: str
    timestamp: float          # event time (when it happened at the device)
    device: str
    event_type: str           # "bgp_notification", "syslog", "snmp_trap", "config_change"
    severity: str             # "critical", "warning", "info"
    message: str
    raw_data: dict = field(default_factory=dict)

    # Enriched fields (added by stream processor)
    device_role: str = ""     # "core", "distribution", "access", "provider-edge"
    device_criticality: int = 0   # 1-5 (5 = most critical)
    affected_services: list = field(default_factory=list)
    incident_id: str = ""


# ── CMDB (Configuration Management Database) — device metadata ─────────────────
CMDB = {
    "core-rtr-01":        {"role": "core",          "criticality": 5, "services": ["internet", "data-center"]},
    "core-rtr-02":        {"role": "core",          "criticality": 5, "services": ["internet", "data-center"]},
    "dist-sw-01":         {"role": "distribution",  "criticality": 4, "services": ["data-center"]},
    "dist-sw-02":         {"role": "distribution",  "criticality": 4, "services": ["data-center"]},
    "access-sw-01":       {"role": "access",        "criticality": 3, "services": ["office-users"]},
    "access-sw-02":       {"role": "access",        "criticality": 3, "services": ["office-users"]},
    "provider-edge-01":   {"role": "provider-edge", "criticality": 5, "services": ["internet"]},
    "provider-edge-02":   {"role": "provider-edge", "criticality": 5, "services": ["internet", "mpls"]},
    "server-01":          {"role": "server",        "criticality": 4, "services": ["app-server"]},
    "branch-fw-01":       {"role": "branch",        "criticality": 3, "services": ["branch-office"]},
}


class EventStreamProcessor:
    '''
    Simulates a stream processor (Apache Flink in production).

    Functions:
    1. Enrich events with CMDB metadata
    2. Group related events into incidents using session windows
    3. Compute anomaly scores to decide if an incident warrants AI triage
    '''
    def __init__(self, session_timeout_s: float = 10.0,
                 anomaly_threshold: float = 3.0):
        self.session_timeout   = session_timeout_s
        self.anomaly_threshold = anomaly_threshold
        self._open_sessions    = {}    # incident_id -> list of events
        self._session_timers   = {}    # incident_id -> last_event_time
        self._closed_incidents = []
        self._event_counts     = defaultdict(int)   # event_type -> count (for anomaly detection)
        self._baseline         = {"bgp_notification": 0.5, "syslog_error": 2.0,
                                  "snmp_trap": 1.0, "config_change": 0.1}  # per minute

    def _enrich(self, event: TelemetryEvent) -> TelemetryEvent:
        'Join event against CMDB to add device context.'
        meta = CMDB.get(event.device, {})
        event.device_role        = meta.get("role", "unknown")
        event.device_criticality = meta.get("criticality", 1)
        event.affected_services  = meta.get("services", [])
        return event

    def _correlate_to_incident(self, event: TelemetryEvent) -> str:
        '''
        Assign event to an open incident session or create a new one.
        Correlation key: events from the same time window with the same event_type
        and shared neighbor/device form one session.
        '''
        # Look for an open session that matches this event type within timeout
        now = event.timestamp
        for iid, last_time in list(self._session_timers.items()):
            if (now - last_time) > self.session_timeout:
                self._close_session(iid)
                continue
            # Check if this event fits the session (same type or related severity)
            session_events = self._open_sessions.get(iid, [])
            if session_events:
                first_event = session_events[0]
                if (first_event.event_type == event.event_type or
                        event.severity in ["critical", "warning"]):
                    self._session_timers[iid] = now
                    return iid

        # Create new session
        iid = f"INC-{len(self._closed_incidents)+len(self._open_sessions)+1:04d}"
        self._open_sessions[iid] = []
        self._session_timers[iid] = now
        return iid

    def _close_session(self, incident_id: str):
        events = self._open_sessions.pop(incident_id, [])
        self._session_timers.pop(incident_id, None)
        if events:
            self._closed_incidents.append({
                "incident_id": incident_id,
                "events":      events,
                "start_time":  events[0].timestamp,
                "end_time":    events[-1].timestamp,
                "device_count": len(set(e.device for e in events)),
                "severity":    max(e.device_criticality for e in events),
                "anomaly_score": self._compute_anomaly(events),
            })

    def _compute_anomaly(self, events: list) -> float:
        'Higher score = more anomalous = more urgent for AI triage.'
        if not events:
            return 0.0
        type_counts = defaultdict(int)
        for e in events:
            type_counts[e.event_type] += 1
        # Compare against baseline rates (events per minute)
        duration_min = max((events[-1].timestamp - events[0].timestamp) / 60, 0.016)
        scores = []
        for etype, count in type_counts.items():
            rate = count / duration_min
            baseline = self._baseline.get(etype, 1.0)
            scores.append(rate / baseline)   # z-score proxy
        return round(max(scores), 2)

    def ingest(self, event: TelemetryEvent) -> str:
        'Process one event. Returns the incident_id it was assigned to.'
        event = self._enrich(event)
        iid = self._correlate_to_incident(event)
        event.incident_id = iid
        self._open_sessions[iid].append(event)
        self._event_counts[event.event_type] += 1
        return iid

    def flush(self):
        'Close all open sessions.'
        for iid in list(self._open_sessions.keys()):
            self._close_session(iid)

    def incidents_requiring_triage(self) -> list:
        return [i for i in self._closed_incidents
                if i["anomaly_score"] >= self.anomaly_threshold]


# ── Simulate the 03:14 BGP incident event flood ───────────────────────────────
random.seed(42)
processor = EventStreamProcessor(session_timeout_s=8.0, anomaly_threshold=2.5)

# The incident: MTU change at 03:12:47, BGP sessions start dropping at 03:14:22
BASE_TIME = 1740960862.0   # 03:14:22 epoch

raw_events = [
    # BGP NOTIFICATION messages (hold-time expiry) from 12 routers
    *[TelemetryEvent(f"EVT-{i+1:03d}", BASE_TIME + i*0.3,
                     f"core-rtr-0{(i%2)+1}", "bgp_notification", "critical",
                     f"BGP NOTIFICATION: hold timer expired, peer 10.0.0.{i+1}",
                     {"notification_code": 6, "peer_ip": f"10.0.0.{i+1}"})
      for i in range(6)],
    *[TelemetryEvent(f"EVT-{i+7:03d}", BASE_TIME + i*0.4 + 0.5,
                     f"dist-sw-0{(i%2)+1}", "bgp_notification", "critical",
                     f"BGP session down: peer provider-edge-02",
                     {"notification_code": 6, "peer": "provider-edge-02"})
      for i in range(4)],
    # Syslog errors
    *[TelemetryEvent(f"EVT-{i+11:03d}", BASE_TIME + i*0.5 + 1.0,
                     random.choice(["core-rtr-01", "core-rtr-02", "dist-sw-01"]),
                     "syslog_error", "warning",
                     f"Interface ge-0/0/{i} experiencing high error rate",
                     {"error_rate": random.randint(50, 200)})
      for i in range(5)],
    # Config change event (the root cause)
    TelemetryEvent("EVT-099", BASE_TIME - 95.0,  # 95 seconds before = 03:12:47
                   "provider-edge-02", "config_change", "info",
                   "Config commit a3f9b2c: MTU change on ge-0/0/3 (9000->1500)",
                   {"commit_id": "a3f9b2c", "change_ticket": "11847", "interface": "ge-0/0/3"}),
]

print("=== Layer 1: Event Stream Processing ===")
print(f"Ingesting {len(raw_events)} telemetry events...")
print()

for event in raw_events:
    iid = processor.ingest(event)

processor.flush()

incidents = processor.incidents_requiring_triage()
print(f"Raw events:                {len(raw_events)}")
print(f"Incidents detected:        {len(processor._closed_incidents)}")
print(f"Requiring AI triage:       {len(incidents)}")
print()

for inc in processor._closed_incidents:
    needs_triage = inc["anomaly_score"] >= 2.5
    print(f"  [{inc['incident_id']}] events={len(inc['events']):>3}, "
          f"devices={inc['device_count']}, "
          f"anomaly={inc['anomaly_score']:>5.1f}, "
          f"severity={inc['severity']}, "
          f"triage={'YES!' if needs_triage else 'no'}")

print()
print("Stream enrichment added (from CMDB):")
sample_event = raw_events[0]
processor._enrich(sample_event)
print(f"  {sample_event.device}: role={sample_event.device_role}, "
      f"criticality={sample_event.device_criticality}, "
      f"services={sample_event.affected_services}")

# Store the primary incident for downstream layers
primary_incident = incidents[0] if incidents else processor._closed_incidents[0]
print(f"\nPrimary incident for AI triage: {primary_incident['incident_id']}")
print(f"  {len(primary_incident['events'])} events, anomaly score={primary_incident['anomaly_score']}")

## Layer 2: Knowledge Graph — Topology + F-score Root Cause Analysis

The knowledge graph models the **dependency relationships** between devices.
When multiple alerts fire, it answers: **"which single failure explains all of these?"**

**F-score Root Cause Analysis:**
- **Precision** = fraction of observed symptoms explained by this candidate root cause
- **Recall** = fraction of this device's known blast radius that is actually showing symptoms
- **F-score** = harmonic mean of precision and recall (0.0 to 1.0)

The candidate with the **highest F-score** is the most probable root cause.

> **In Simple Words:** F-score RCA is like running OSPF SPF — but instead of finding
> the shortest path from source to all destinations, you find the most probable failure
> origin from all observed symptoms. Both are graph traversal with a cost metric.
>
> SPOF detection = any node where multiple dependency paths converge with no alternative.
> "If this goes down, everything downstream goes down." — never leave SPOFs unresolved.

In [ ]:
# ── Layer 2: Knowledge Graph + F-score Root Cause Analysis ────────────────────

class NetworkGraph:
    '''
    Adjacency-list property graph representing network topology.
    Nodes = devices, Edges = physical/logical connections.
    Used for: dependency traversal, blast radius, SPOF detection, RCA.
    '''
    def __init__(self):
        self._nodes = {}   # device -> {role, criticality, services, ...}
        self._edges = defaultdict(list)   # device -> [(neighbor, link_type, props)]

    def add_node(self, device: str, **props):
        self._nodes[device] = props

    def add_edge(self, a: str, b: str, link_type: str = "physical", **props):
        self._edges[a].append((b, link_type, props))
        self._edges[b].append((a, link_type, props))   # undirected

    def neighbors(self, device: str) -> list:
        return [(n, lt) for n, lt, _ in self._edges.get(device, [])]

    def blast_radius(self, failed_device: str, max_depth: int = 3) -> set:
        'BFS: find all devices that lose connectivity if failed_device goes down.'
        visited = set()
        queue = deque([(failed_device, 0)])
        while queue:
            node, depth = queue.popleft()
            if node in visited or depth > max_depth:
                continue
            visited.add(node)
            for neighbor, _ in self.neighbors(node):
                if neighbor not in visited:
                    queue.append((neighbor, depth + 1))
        visited.discard(failed_device)
        return visited

    def find_spofs(self) -> list:
        'Find single points of failure: nodes whose removal disconnects others.'
        spofs = []
        all_devices = list(self._nodes.keys())
        # Simplified: a node is SPOF if removing it strands any reachable neighbor
        for device in all_devices:
            neighbors_of = [n for n, _ in self.neighbors(device)]
            if len(neighbors_of) > 2 and self._nodes.get(device, {}).get("criticality", 0) >= 4:
                spofs.append(device)
        return spofs


def f_score_rca(symptoms: list, candidate: str, graph: NetworkGraph) -> dict:
    '''
    F-score Root Cause Analysis.

    precision = |symptoms explained by candidate| / |all symptoms|
    recall    = |symptoms matching candidate blast radius| / |full blast radius|
    f_score   = harmonic mean (2 * P * R) / (P + R)
    '''
    blast = graph.blast_radius(candidate, max_depth=3)
    blast_with_self = blast | {candidate}

    explained = set(s for s in symptoms if s in blast_with_self)
    precision = len(explained) / len(symptoms) if symptoms else 0.0
    recall    = len(explained) / len(blast_with_self) if blast_with_self else 0.0
    f = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "candidate":    candidate,
        "precision":    round(precision, 3),
        "recall":       round(recall, 3),
        "f_score":      round(f, 3),
        "blast_radius": len(blast),
        "explained":    sorted(explained),
    }


# ── Build the network topology graph ──────────────────────────────────────────
G = NetworkGraph()

# Add nodes
for device, meta in CMDB.items():
    G.add_node(device, **meta)

# Add edges (physical/BGP connections)
edges = [
    ("core-rtr-01",    "provider-edge-01", "bgp"),
    ("core-rtr-01",    "provider-edge-02", "bgp"),
    ("core-rtr-02",    "provider-edge-01", "bgp"),
    ("core-rtr-02",    "provider-edge-02", "bgp"),
    ("core-rtr-01",    "dist-sw-01",       "ospf"),
    ("core-rtr-01",    "dist-sw-02",       "ospf"),
    ("core-rtr-02",    "dist-sw-01",       "ospf"),
    ("core-rtr-02",    "dist-sw-02",       "ospf"),
    ("dist-sw-01",     "access-sw-01",     "physical"),
    ("dist-sw-01",     "access-sw-02",     "physical"),
    ("dist-sw-02",     "access-sw-01",     "physical"),
    ("dist-sw-02",     "server-01",        "physical"),
    ("branch-fw-01",   "dist-sw-02",       "vpn"),
]
for a, b, lt in edges:
    G.add_edge(a, b, link_type=lt)

# ── Observed symptoms from the incident ───────────────────────────────────────
SYMPTOMS = ["core-rtr-01", "core-rtr-02", "dist-sw-01", "dist-sw-02",
            "access-sw-01", "access-sw-02", "server-01"]

# ── Run F-score RCA against candidate root causes ─────────────────────────────
candidates = list(CMDB.keys())

print("=== Layer 2: F-score Root Cause Analysis ===")
print(f"Observed symptoms: {SYMPTOMS}")
print(f"Running RCA against {len(candidates)} candidate devices...")
print()

rca_results = [f_score_rca(SYMPTOMS, c, G) for c in candidates]
rca_results.sort(key=lambda x: x["f_score"], reverse=True)

print(f"{'Candidate':<22} {'Precision':>10} {'Recall':>8} {'F-score':>8} "
      f"{'Blast R':>8} {'Verdict'}")
print("-" * 75)
for r in rca_results:
    verdict = ""
    if r["f_score"] >= 0.70:
        verdict = "<-- PROBABLE ROOT CAUSE"
    elif r["f_score"] >= 0.40:
        verdict = "<-- possible"
    print(f"{r['candidate']:<22} {r['precision']:>10.3f} {r['recall']:>8.3f} "
          f"{r['f_score']:>8.3f} {r['blast_radius']:>8}   {verdict}")

top_candidate = rca_results[0]
print()
print(f"TOP ROOT CAUSE: {top_candidate['candidate']}")
print(f"  F-score:      {top_candidate['f_score']:.3f} (precision={top_candidate['precision']:.3f}, recall={top_candidate['recall']:.3f})")
print(f"  Explains:     {top_candidate['explained']}")
print(f"  Blast radius: {top_candidate['blast_radius']} downstream devices")

# ── SPOF detection ────────────────────────────────────────────────────────────
spofs = G.find_spofs()
print()
print("=== SPOF Detection ===")
print(f"Detected {len(spofs)} single points of failure:")
for spof in spofs:
    meta = CMDB.get(spof, {})
    br = G.blast_radius(spof)
    print(f"  {spof} (role={meta.get('role','?')}, criticality={meta.get('criticality','?')}, "
          f"blast_radius={len(br)} devices)")
    print(f"  -> Services at risk: {meta.get('services', [])}")

# Store top RCA for downstream layers
RCA_RESULT = top_candidate
print(f"\nRCA complete. Feeding '{top_candidate['candidate']}' to semantic layer...")

## Layer 3: Semantic Layer — RAG Runbook Retrieval

The knowledge graph gives structured topology data.
The **RAG layer** adds unstructured expertise: runbooks, vendor advisories, historical fixes.

When BGP sessions drop on `provider-edge-02`, the RAG system retrieves:
- "BGP session instability on PE routers: check MTU on transit links"
- "hold-timer expiry pattern linked to jumbo frame MTU mismatch"
- "change-ticket-11847: automated provisioning changed interface MTU"

**Without RAG context:** AI knows BGP theory, guesses generic causes.
**With RAG context:** AI knows YOUR network's history and documented patterns.

> **In Simple Words:** RAG is your institutional memory in a searchable format.
> A new NOC engineer gets the same knowledge as a 10-year veteran — instantly.
> The AI doesn't just know BGP theory; it knows BGP on YOUR devices.

In [ ]:
# ── Layer 3: Semantic RAG Layer ───────────────────────────────────────────────

# ── Network knowledge base (runbooks, advisories, incident history) ────────────
KNOWLEDGE_BASE = [
    {
        "id": "KB-001",
        "title": "BGP Session Instability — Hold Timer Expiry",
        "content": ("BGP sessions drop with NOTIFICATION code 6 (cease) when hold timer expires. "
                    "Common causes: (1) MTU mismatch causing BGP UPDATE fragmentation, "
                    "(2) keepalive interval too long, (3) high CPU causing missed keepalives. "
                    "Check: MTU on all links between BGP peers, verify keepalive/hold timers match."),
        "tags": ["bgp", "hold-timer", "notification", "mtu"]
    },
    {
        "id": "KB-002",
        "title": "MTU Mismatch on Provider-Edge Routers",
        "content": ("Provider-edge routers require MTU 9000 (jumbo frames) on transit interfaces. "
                    "If MTU is reduced below 9000, BGP UPDATE messages exceeding 1500 bytes get "
                    "fragmented or dropped, causing hold-timer expiry. "
                    "Resolution: set interface MTU to 9000 on both sides of the transit link."),
        "tags": ["mtu", "jumbo-frames", "provider-edge", "bgp", "transit"]
    },
    {
        "id": "KB-003",
        "title": "Automated Provisioning MTU Changes (change-ticket-11847 pattern)",
        "content": ("Automated provisioning script version < 2.3 incorrectly sets MTU to 1500 "
                    "on provider-edge interfaces when deploying new VLAN configurations. "
                    "If MTU change correlates with BGP session drops, check recent provisioning "
                    "commits and revert MTU to 9000. Fixed in provisioning script v2.3."),
        "tags": ["automation", "provisioning", "mtu", "bug", "change-management"]
    },
    {
        "id": "KB-004",
        "title": "OSPF Area Type Migration Runbook",
        "content": ("When converting an area from standard to stub: (1) verify no ASBR in area, "
                    "(2) set stub on all routers simultaneously, (3) check that default route is "
                    "advertised into stub area from ABR. Rolling back: remove stub keyword "
                    "and wait for LSA refresh (30 minutes)."),
        "tags": ["ospf", "stub", "area", "migration"]
    },
    {
        "id": "KB-005",
        "title": "QoS DSCP Marking Policy for Voice Traffic",
        "content": ("Voice traffic must be marked DSCP EF (46) at ingress. "
                    "Interactive video: AF41. Call signaling: CS3. "
                    "Verify markings survive across provider links — some providers re-mark traffic. "
                    "Use 'show policy-map interface' to verify drops in EF queue."),
        "tags": ["qos", "dscp", "voice", "ef", "marking"]
    },
    {
        "id": "KB-006",
        "title": "BGP ECMP Load Balancing Configuration",
        "content": ("BGP ECMP requires: maximum-paths N (same for iBGP and eBGP), "
                    "multipath same-cluster-length (for iBGP across route reflectors). "
                    "Verify all ECMP paths have identical AS_PATH and other attributes. "
                    "Check CEF adjacency table with 'show ip cef <prefix> detail'."),
        "tags": ["bgp", "ecmp", "load-balancing", "multipath"]
    },
    {
        "id": "KB-007",
        "title": "BGP Route Leak Detection and Response",
        "content": ("Route leak indicators: unexpected prefixes in BGP table, traffic redirection, "
                    "sudden AS_PATH lengthening. Response: (1) identify leaking AS, "
                    "(2) apply inbound prefix-list to filter leaked prefixes, "
                    "(3) contact upstream provider. Use BGP monitoring services."),
        "tags": ["bgp", "route-leak", "security", "prefix-list"]
    },
    {
        "id": "KB-008",
        "title": "Interface Error Counter Thresholds",
        "content": ("Alert thresholds: input errors >0.1% of input packets, "
                    "CRC errors >10 per minute, output drops >100 per minute. "
                    "CRC errors indicate physical layer issues (cable, SFP, duplex mismatch). "
                    "Output drops indicate congestion — check QoS policy and interface speed."),
        "tags": ["interface", "errors", "crc", "monitoring", "thresholds"]
    },
]


def tokenize(text: str) -> list:
    return re.findall(r'[a-z0-9]+', text.lower())

def tfidf_embed(text: str, vocab: dict) -> list:
    tokens = tokenize(text)
    vec = [0.0] * len(vocab)
    for t in tokens:
        if t in vocab:
            vec[vocab[t]] += 1.0
    mag = math.sqrt(sum(x*x for x in vec))
    return [x/mag for x in vec] if mag > 0 else vec

def cosine(a: list, b: list) -> float:
    min_len = min(len(a), len(b))
    if min_len == 0:
        return 0.0
    return sum(a[i]*b[i] for i in range(min_len))


class RAGRetriever:
    '''
    Semantic retrieval using TF-IDF embeddings + cosine similarity.
    No external libraries - pure Python.
    In production: replace with Voyage AI embeddings + pgvector.
    '''
    def __init__(self, knowledge_base: list):
        self._kb = knowledge_base
        # Build vocabulary from all documents
        all_words = []
        for doc in knowledge_base:
            all_words.extend(tokenize(doc["title"] + " " + doc["content"]))
        unique = sorted(set(all_words))
        self._vocab = {w: i for i, w in enumerate(unique)}
        # Pre-embed all documents
        self._embeddings = []
        for doc in knowledge_base:
            text = doc["title"] + " " + doc["content"] + " " + " ".join(doc.get("tags", []))
            self._embeddings.append(tfidf_embed(text, self._vocab))

    def retrieve(self, query: str, top_k: int = 3) -> list:
        q_vec = tfidf_embed(query, self._vocab)
        scored = []
        for i, (doc, emb) in enumerate(zip(self._kb, self._embeddings)):
            score = cosine(q_vec, emb)
            scored.append((score, doc))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [(score, doc) for score, doc in scored[:top_k]]

    def build_context(self, query: str, top_k: int = 3) -> str:
        results = self.retrieve(query, top_k)
        context_parts = []
        for score, doc in results:
            context_parts.append(f"[{doc['id']}] {doc['title']} (relevance={score:.3f}):\n{doc['content']}")
        return "\n\n".join(context_parts)


# ── Retrieve context for the BGP incident ─────────────────────────────────────
retriever = RAGRetriever(KNOWLEDGE_BASE)

# The query is built from the incident + RCA findings
incident_query = (f"BGP sessions dropping hold timer expiry provider-edge MTU "
                  f"notification code 6 automated provisioning change {RCA_RESULT['candidate']}")

print("=== Layer 3: RAG Runbook Retrieval ===")
print(f"Query built from incident + RCA findings:")
print(f"  '{incident_query[:80]}...'")
print()

results = retriever.retrieve(incident_query, top_k=4)
print(f"Top {len(results)} retrieved documents:")
print()
for score, doc in results:
    print(f"  [{doc['id']}] Score={score:.4f}: {doc['title']}")
    print(f"  Tags: {doc['tags']}")
    print(f"  Content (first 120 chars): {doc['content'][:120]}...")
    print()

# Build the full context for AI
rag_context = retriever.build_context(incident_query, top_k=3)
print("RAG context ready for AI agents (top-3 documents combined):")
print(f"  Total context length: {len(rag_context)} characters")
print(f"  Approx tokens: ~{len(rag_context)//4}")

# Store for downstream
RAG_CONTEXT = rag_context

## Layer 4: Multi-Agent Triage — Supervisor + Specialists + HITL

Three specialist agents, one supervisor:

```
Supervisor Agent
    |-- Diagnosis Agent     (graph + RAG -> root cause)
    |-- Config Agent        (git diff -> what changed)
    `-- Performance Agent   (NetFlow + SNMP -> traffic impact)
        |
        v
    Synthesis -> Incident Report
        |
        v
    HITL Approval (Human-in-the-Loop)
        |
        v
    Execute Fix  OR  Escalate to engineer
```

**Change Risk Assessment** drives the HITL threshold:
- Reversible + narrow blast radius + known pattern = eligible for auto-execution
- Irreversible OR wide blast radius OR novel = **always require human approval**

> **In Simple Words:** This is your Tier 1 / Tier 2 NOC escalation model,
> but automated. Tier 1 (Supervisor) classifies and routes. Tier 2 specialists
> do deep analysis. The engineer sees a pre-built runbook with evidence —
> not raw logs. One button click = approved. 53 seconds instead of 3 hours.

In [ ]:
# ── Layer 4: Multi-Agent Triage Orchestrator ─────────────────────────────────

@dataclass
class AgentResult:
    agent_name: str
    finding: str
    confidence: float
    evidence: list
    latency_ms: float = 0.0


class SpecialistAgent:
    '''
    Base specialist agent: receives a task, calls LLM with specific system prompt,
    returns structured AgentResult.
    '''
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt

    def run(self, task: str, context: str = "") -> AgentResult:
        start = time.time()
        full_input = f"Task: {task}\nContext:\n{context[:800]}" if context else f"Task: {task}"
        raw = llm(full_input, system=self.system_prompt, max_tokens=250)
        lat = (time.time() - start) * 1000
        # Parse JSON result or gracefully degrade
        try:
            start_j = raw.find('{')
            end_j   = raw.rfind('}') + 1
            if start_j >= 0 and end_j > start_j:
                data = json.loads(raw[start_j:end_j])
                return AgentResult(
                    agent_name=self.name,
                    finding=data.get("finding", raw),
                    confidence=float(data.get("confidence", 0.5)),
                    evidence=data.get("evidence", []),
                    latency_ms=round(lat, 1),
                )
        except (json.JSONDecodeError, ValueError):
            pass
        return AgentResult(self.name, raw[:200], 0.5, [], round(lat, 1))


def assess_change_risk(finding: str, evidence: list) -> dict:
    '''
    Change risk assessment: reversible x blast_radius x novelty.
    Low scores = eligible for autonomous execution.
    High scores = require human approval.
    '''
    f = finding.lower()
    # Reversibility
    reversible = 3 if any(w in f for w in ["revert", "restore", "rollback", "mtu"]) else 1
    # Blast radius
    if any(w in f for w in ["core", "12 session", "all", "internet"]):
        blast = 3
    elif any(w in f for w in ["distribution", "multiple"]):
        blast = 2
    else:
        blast = 1
    # Novelty (seen this pattern before?)
    novelty = 1 if len(evidence) >= 3 else 2   # more evidence = less novel

    risk_score = reversible + blast + novelty
    if risk_score <= 5:
        action = "APPROVE_AUTONOMOUS"
        reason = "Reversible, known pattern, evidence-backed"
    elif risk_score <= 7:
        action = "REQUIRE_HUMAN_APPROVAL"
        reason = "Medium risk — human review recommended"
    else:
        action = "ESCALATE_TO_SENIOR"
        reason = "High risk — senior engineer required"

    return {"risk_score": risk_score, "action": action, "reason": reason,
            "reversible": reversible, "blast": blast, "novelty": novelty}


class SupervisorAgent:
    '''
    Orchestrates the investigation:
    1. Plan which specialists to invoke
    2. Run specialists (sequentially or in parallel)
    3. Synthesize findings
    4. Assess risk and recommend action
    5. Generate HITL approval request
    '''
    def __init__(self):
        self.specialists = {
            "bgp_diagnosis": SpecialistAgent(
                "BGP Diagnosis",
                'You are a BGP expert. Analyze the incident and return JSON only: '
                '{"finding": str, "confidence": 0.0-1.0, "evidence": [str]}'
            ),
            "config_analysis": SpecialistAgent(
                "Config Analysis",
                'You are a network config analyst. Find configuration changes causing this issue. '
                'Return JSON only: {"finding": str, "confidence": 0.0-1.0, "evidence": [str]}'
            ),
            "topology_impact": SpecialistAgent(
                "Topology Impact",
                'You are a network topology expert. Assess traffic impact and affected services. '
                'Return JSON only: {"finding": str, "confidence": 0.0-1.0, "evidence": [str]}'
            ),
        }

    def plan(self, alert: str) -> list:
        raw = llm(f"Alert: {alert}", system=(
            'You are a network incident supervisor. Decide which investigations to run. '
            'Output JSON only: {"investigations": ["bgp_diagnosis","config_analysis","topology_impact"]} '
            'Choose only relevant ones.'
        ), max_tokens=100)
        try:
            data = json.loads(raw[raw.find('{'):raw.rfind('}')+1])
            return data.get("investigations", ["bgp_diagnosis"])
        except (json.JSONDecodeError, ValueError):
            return ["bgp_diagnosis", "config_analysis"]

    def investigate(self, plan: list, context: str) -> list:
        results = []
        for investigation in plan:
            agent = self.specialists.get(investigation)
            if not agent:
                continue
            print(f"  [{investigation}] running...")
            result = agent.run(f"Investigate: {investigation}", context=context)
            results.append(result)
            print(f"  [{investigation}] confidence={result.confidence:.2f}, "
                  f"latency={result.latency_ms:.0f}ms")
        return results

    def synthesize(self, alert: str, results: list, rag_ctx: str,
                   rca: dict) -> dict:
        findings_text = "\n".join(
            f"- {r.agent_name}: {r.finding} (confidence={r.confidence:.2f})"
            for r in results
        )
        synthesis_prompt = (
            f"Alert: {alert}\n"
            f"RCA Result: {rca['candidate']} (F-score={rca['f_score']})\n"
            f"Agent Findings:\n{findings_text}\n"
            f"Relevant Runbooks:\n{rag_ctx[:500]}"
        )
        raw = llm(synthesis_prompt, system=(
            'Synthesize into a brief incident report with: Root Cause, Impact, '
            'Proposed Fix, Risk level (LOW/MEDIUM/HIGH).'
        ), max_tokens=350)

        # Risk assessment
        all_evidence = []
        for r in results:
            all_evidence.extend(r.evidence)
        avg_confidence = sum(r.confidence for r in results) / len(results) if results else 0.5
        risk = assess_change_risk(raw, all_evidence)

        return {
            "incident_report":  raw,
            "avg_confidence":   round(avg_confidence, 3),
            "risk_assessment":  risk,
            "evidence_count":   len(all_evidence),
            "specialists_run":  [r.agent_name for r in results],
        }

    def generate_hitl_request(self, synthesis: dict, alert: str) -> dict:
        'Generate the Slack/PagerDuty HITL approval message.'
        risk = synthesis["risk_assessment"]
        return {
            "type":           "HITL_APPROVAL_REQUEST",
            "message":        f"BGP Incident — AI Triage Complete",
            "confidence":     synthesis["avg_confidence"],
            "evidence_count": synthesis["evidence_count"],
            "risk":           risk["action"],
            "risk_reason":    risk["reason"],
            "summary":        synthesis["incident_report"][:300],
            "buttons":        ["[APPROVE]", "[REJECT]", "[ESCALATE]"],
            "warning":        "Review evidence before approving. AI recommendations are not infallible.",
        }


# ── Run the full triage ────────────────────────────────────────────────────────
alert_text = (
    "BGP sessions dropping on 12 devices. All share neighbor provider-edge-02. "
    "NOTIFICATION code 6 (hold-timer expiry). Config diff shows MTU change at 03:12:47. "
    f"RCA: {RCA_RESULT['candidate']} (F-score={RCA_RESULT['f_score']})."
)
full_context = f"{alert_text}\n\nKnowledge Base:\n{RAG_CONTEXT[:600]}"

print("=== Layer 4: Multi-Agent Triage ===")
print(f"Alert: {alert_text[:100]}...")
print()

supervisor = SupervisorAgent()

# Step 1: Plan
print("Step 1: Supervisor plans investigations...")
investigation_plan = supervisor.plan(alert_text)
print(f"  Plan: {investigation_plan}")
print()

# Step 2: Run specialists
print("Step 2: Running specialist agents...")
specialist_results = supervisor.investigate(investigation_plan, full_context)
print()

# Step 3: Synthesize
print("Step 3: Synthesizing findings...")
synthesis = supervisor.synthesize(alert_text, specialist_results, RAG_CONTEXT, RCA_RESULT)

# Step 4: HITL
hitl = supervisor.generate_hitl_request(synthesis, alert_text)

print()
print("=== INCIDENT REPORT ===")
print(synthesis["incident_report"])
print()
print("=== HITL APPROVAL REQUEST ===")
print(f"  Message:     {hitl['message']}")
print(f"  Confidence:  {hitl['confidence']:.3f}")
print(f"  Evidence:    {hitl['evidence_count']} data points")
print(f"  Risk:        {hitl['risk']}")
print(f"  Risk reason: {hitl['risk_reason']}")
print(f"  Action:      {hitl['buttons']}")
print(f"  Warning:     {hitl['warning']}")

# Store for final layer
SYNTHESIS    = synthesis
HITL_REQUEST = hitl

## Layer 5: Full Incident Lifecycle — 56 Seconds from Alert to Fix

Everything wired together. The BGP incident at 3:14 AM:

```
03:14:22  12 routers lose BGP sessions (events flood into Kafka)
03:14:28  Stream processor: session window closes, anomaly score=8.4 -> AI triage
03:14:35  Supervisor Agent activated, plans investigations
03:14:41  Diagnosis: provider-edge-02 (F-score=0.89)
03:14:47  RAG: retrieves KB-002, KB-003 (MTU runbooks)
03:14:52  Config Agent: commit a3f9b2c, MTU 9000->1500 at 03:12:47
03:14:58  Synthesis + HITL request sent to PagerDuty/Slack
03:15:01  On-call engineer paged
03:15:15  Engineer reads evidence, clicks [APPROVE]
03:15:18  Fix applied, BGP sessions recover
```

**56 seconds total. Without AI: 3 hours.**

> **Crawl-Walk-Run Framework for Trust:**
> - **Crawl** (months 1-3): AI recommends, human always approves + executes
> - **Walk** (months 4-6): AI executes low-risk reversible fixes autonomously
> - **Run** (months 7+): AI manages full incident classes; human "on the loop"
>
> Always show reasoning chain, not just conclusion.
> The engineer can verify evidence and build correct trust.

In [ ]:
# ── Layer 5: Full Incident Lifecycle — End-to-End 56-Second Scenario ──────────

@dataclass
class IncidentTicket:
    incident_id: str
    alert_text: str
    created_at: float
    events: list
    rca_candidate: str
    rca_f_score: float
    rag_docs_retrieved: int
    agents_invoked: list
    avg_confidence: float
    risk_action: str
    status: str = "open"     # open -> triaged -> approved -> executing -> resolved
    resolved_at: Optional[float] = None
    resolution: Optional[str] = None
    hitl_approved_by: Optional[str] = None

    def elapsed_s(self, from_time: Optional[float] = None) -> float:
        t = from_time or self.created_at
        if self.resolved_at:
            return round(self.resolved_at - t, 1)
        return round(time.time() - t, 1)

    def mttr_s(self) -> Optional[float]:
        return self.elapsed_s() if self.resolved_at else None


class NetOpsAIPlatform:
    '''
    Complete NetOps AI Platform — all 5 layers assembled.

    Components:
      Layer 1: EventStreamProcessor (telemetry ingestion + windowing)
      Layer 2: NetworkGraph + f_score_rca (topology + RCA)
      Layer 3: RAGRetriever (semantic runbook retrieval)
      Layer 4: SupervisorAgent (multi-agent orchestration)
      Layer 5: IncidentTicket lifecycle + HITL + execution

    In production:
      Layer 1: Apache Kafka + Flink
      Layer 2: Neo4j or custom graph DB
      Layer 3: pgvector + Voyage AI embeddings
      Layer 4: LangGraph with MCP tools
      Layer 5: PagerDuty + Slack HITL + network device APIs
    '''
    def __init__(self, graph: NetworkGraph, knowledge_base: list):
        self.stream    = EventStreamProcessor(session_timeout_s=8.0, anomaly_threshold=2.5)
        self.graph     = graph
        self.retriever = RAGRetriever(knowledge_base)
        self.supervisor = SupervisorAgent()
        self._tickets  = []
        self._timeline = []   # (timestamp_offset, event_label)

    def _log(self, offset_s: float, label: str):
        self._timeline.append((offset_s, label))
        print(f"  t+{offset_s:5.2f}s  {label}")

    def handle_incident(self, events: list, sim_start: float = 0.0) -> IncidentTicket:
        t = sim_start

        # ── Layer 1: Ingest events ────────────────────────────────────────────
        self._log(0.0, "Events flood in from 12 routers (BGP NOTIFICATION x12)")
        for e in events:
            self.stream.ingest(e)
        self.stream.flush()
        incidents = self.stream.incidents_requiring_triage()
        if not incidents:
            incidents = self.stream._closed_incidents

        inc_data = incidents[0] if incidents else self.stream._closed_incidents[0]
        self._log(5.8, f"Stream processor: anomaly_score={inc_data['anomaly_score']} "
                       f"-> {inc_data['incident_id']} created, AI triage triggered")

        # Build alert text from event summary
        alert = (f"BGP sessions dropping on {inc_data['device_count']} devices. "
                 f"{len(inc_data['events'])} correlated events in session window. "
                 f"Severity={inc_data['severity']}. Anomaly score={inc_data['anomaly_score']}.")

        ticket = IncidentTicket(
            incident_id=inc_data["incident_id"],
            alert_text=alert,
            created_at=time.time(),
            events=inc_data["events"],
            rca_candidate="", rca_f_score=0.0,
            rag_docs_retrieved=0,
            agents_invoked=[],
            avg_confidence=0.0,
            risk_action=""
        )

        # ── Layer 2: RCA ──────────────────────────────────────────────────────
        self._log(12.8, "Supervisor Agent activated -> dispatching Diagnosis Agent")
        candidates = list(CMDB.keys())
        symptoms = list(set(e.device for e in inc_data["events"]))
        rca_results = [f_score_rca(symptoms, c, self.graph) for c in candidates]
        rca_results.sort(key=lambda x: x["f_score"], reverse=True)
        top = rca_results[0]
        ticket.rca_candidate = top["candidate"]
        ticket.rca_f_score   = top["f_score"]
        self._log(19.4, f"Knowledge graph RCA: {top['candidate']} "
                        f"(F-score={top['f_score']:.2f}, explains {len(top['explained'])} symptoms)")
        ticket.status = "triaged"

        # ── Layer 3: RAG ──────────────────────────────────────────────────────
        query = f"BGP hold timer expiry MTU mismatch {top['candidate']}"
        rag_results = self.retriever.retrieve(query, top_k=3)
        ticket.rag_docs_retrieved = len(rag_results)
        rag_ctx = self.retriever.build_context(query, top_k=3)
        doc_ids = [d["id"] for _, d in rag_results]
        self._log(25.3, f"RAG retrieval: {doc_ids} (MTU runbooks, provisioning bug)")

        # ── Layer 4: Multi-Agent ──────────────────────────────────────────────
        full_ctx = f"{alert}\nRCA: {top['candidate']} F={top['f_score']}\n{rag_ctx[:500]}"
        inv_plan = self.supervisor.plan(alert)
        self._log(30.1, f"Supervisor plans: {inv_plan}")

        spec_results = self.supervisor.investigate(inv_plan, full_ctx)
        for r in spec_results:
            self._log(33.0 + spec_results.index(r) * 6,
                      f"{r.agent_name}: confidence={r.confidence:.2f}")
        ticket.agents_invoked = [r.agent_name for r in spec_results]

        # ── Synthesis ─────────────────────────────────────────────────────────
        synthesis = self.supervisor.synthesize(alert, spec_results, rag_ctx, top)
        ticket.avg_confidence = synthesis["avg_confidence"]
        ticket.risk_action    = synthesis["risk_assessment"]["action"]
        self._log(52.7, f"Incident report synthesized (confidence={synthesis['avg_confidence']:.2f})")

        # ── HITL ──────────────────────────────────────────────────────────────
        hitl = self.supervisor.generate_hitl_request(synthesis, alert)
        self._log(56.0, f"PagerDuty alert fired -> on-call engineer paged")
        self._log(70.2, "Engineer reads evidence (53 seconds)")
        self._log(72.8, f"Engineer clicks [APPROVE] (risk={hitl['risk']})")
        ticket.hitl_approved_by = "on-call-engineer@company.com"
        ticket.status = "approved"

        # ── Execution ─────────────────────────────────────────────────────────
        self._log(73.2, "Executing fix: set MTU 9000 on provider-edge-02 ge-0/0/3")
        self._log(75.5, "Verifying: BGP sessions recovering...")
        self._log(76.8, "BGP sessions UP on 12/12 devices. Incident resolved.")

        ticket.status     = "resolved"
        ticket.resolved_at = time.time()
        ticket.resolution  = ("Reverted MTU to 9000 on provider-edge-02 ge-0/0/3. "
                               "BGP sessions recovered. Root cause: automated provisioning "
                               "bug in script v<2.3 (change-ticket-11847).")

        self._tickets.append(ticket)
        return ticket

    def print_timeline(self):
        print("\n=== INCIDENT TIMELINE ===")
        for offset, label in self._timeline:
            bar = "#" * min(int(offset / 3), 25)
            print(f"  t+{offset:5.1f}s  {bar}  {label}")

    def metrics(self) -> dict:
        resolved = [t for t in self._tickets if t.resolved_at]
        if not resolved:
            return {}
        mttrs = [t.mttr_s() for t in resolved]
        return {
            "incidents_handled": len(self._tickets),
            "incidents_resolved": len(resolved),
            "avg_mttr_s":  round(sum(mttrs)/len(mttrs), 1),
            "min_mttr_s":  round(min(mttrs), 1),
            "api_calls":   CALL_COUNT[0],
        }


# ── Run the full 56-second scenario ───────────────────────────────────────────
print("=" * 65)
print("NETOPS AI PLATFORM — FULL INCIDENT LIFECYCLE")
print("=" * 65)
print()
print("Scenario: BGP incident at 03:14:22")
print("12 routers lose sessions. MTU change at 03:12:47 is the root cause.")
print()
print("Platform response:")
print()

platform = NetOpsAIPlatform(G, KNOWLEDGE_BASE)
ticket = platform.handle_incident(raw_events)

platform.print_timeline()

print()
print("=== INCIDENT TICKET SUMMARY ===")
print(f"  Incident ID:        {ticket.incident_id}")
print(f"  Status:             {ticket.status}")
print(f"  MTTR:               {ticket.mttr_s():.1f} seconds")
print(f"  Root cause:         {ticket.rca_candidate} (F-score={ticket.rca_f_score:.2f})")
print(f"  RAG docs retrieved: {ticket.rag_docs_retrieved}")
print(f"  Agents invoked:     {ticket.agents_invoked}")
print(f"  AI confidence:      {ticket.avg_confidence:.3f}")
print(f"  Risk assessment:    {ticket.risk_action}")
print(f"  Approved by:        {ticket.hitl_approved_by}")
print(f"  Resolution:         {ticket.resolution}")
print()
print("=== PLATFORM METRICS ===")
m = platform.metrics()
print(f"  Incidents handled:  {m['incidents_handled']}")
print(f"  MTTR:               {m['avg_mttr_s']}s  (vs 180 minutes manually = 99.6% faster)")
print(f"  LLM API calls:      {m['api_calls']}")
print(f"  Estimated cost:     ${m['api_calls'] * 0.0015:.4f}  (vs 3h of engineer time)")
print()
print("=" * 65)
print("COMPARISON: WITH vs WITHOUT NetOps AI")
print("=" * 65)
comparison = [
    ("Event detection",      "8 minutes (NOC dashboard alarm)",      "5.8 seconds (stream processor)"),
    ("Root cause analysis",  "60-90 minutes (manual log grep)",      "19.4 seconds (F-score graph RCA)"),
    ("Runbook retrieval",    "10-20 minutes (search wiki/confluence)","25.3 seconds (RAG retrieval)"),
    ("Fix recommendation",   "30-60 minutes (senior engineer)",      "53 seconds (multi-agent)"),
    ("MTTR total",           "180 minutes (3 hours)",                 "76.8 seconds"),
    ("AI API cost",          "$0",                                    f"${m['api_calls']*0.0015:.4f}"),
]
print(f"{'Phase':<28} {'Without AI':>30} {'With AI':>25}")
print("-" * 85)
for phase, without, with_ai in comparison:
    print(f"{phase:<28} {without:>30} {with_ai:>25}")

## Summary — What You Built

This is the synthesis of the entire book.

| Layer | Component | Chapter Reference |
|-------|-----------|-------------------|
| 1. Data Integration | Event stream, session windowing, enrichment | Ch 48 (Monitoring) |
| 2. Knowledge Graph | F-score RCA, SPOF detection, blast radius | Ch 37 (Graph RAG) |
| 3. Semantic Layer | RAG retrieval, runbook retrieval, context | Ch 35-36 (Vector DB, RAG) |
| 4. Multi-Agent | Supervisor, specialist agents, HITL | Ch 34 (Multi-Agent), Ch 38 (MCP) |
| 5. Lifecycle | Queue, scaling, caching, observability | Ch 40, 48, 51 |

### The Numbers
| Metric | Without AI | With AI | Improvement |
|--------|-----------|---------|-------------|
| MTTD | 8 minutes | 6 seconds | 80x faster |
| MTTR | 3 hours | 77 seconds | 140x faster |
| API Cost | $0 | ~$0.005/incident | Fraction of 1 engineer-hour |

### The Crawl-Walk-Run Trust Framework
```
Crawl (months 1-3):  AI recommends -> human always approves + executes
Walk  (months 4-6):  AI executes low-risk reversible fixes autonomously
Run   (months 7+):   AI manages full incident classes; human "on the loop"
```

### What AI Does NOT Replace
AI eliminates the **manual correlation work** — grep-through-syslog, compare-configs,
find-the-change-in-ServiceNow.

The **engineer's judgment** applies at the decision point:
- Was the RCA reasoning correct?
- Is this fix safe in this specific context?
- Does the proposed change account for factors not in the logs?

**Deep network expertise + AI engineering = production NetOps AI that works.**

### Volume 4 Preview
The next volume covers: Agentic AI beyond single networks — federated NOC AI across
multiple organizations, privacy-preserving federated learning for network anomaly detection,
and the emerging standard for AI-to-AI network operation protocols.

**Stay tuned!**

In [ ]:
# ── Production Deployment Checklist ──────────────────────────────────────────

print("=" * 65)
print("PRODUCTION DEPLOYMENT CHECKLIST")
print("=" * 65)
print()

checklist = [
    # (component, production_equivalent, status)
    ("Event Stream",         "Apache Kafka + Flink with event-time watermarks",          "required"),
    ("Knowledge Graph DB",   "Neo4j or custom graph DB with CMDB sync",                  "required"),
    ("RAG Embeddings",       "Voyage AI embeddings + pgvector or Pinecone",              "required"),
    ("Agent Framework",      "LangGraph with MCP tool registration + checkpointing",     "required"),
    ("HITL Interface",       "Slack bot with [Approve]/[Reject] buttons + PagerDuty",    "required"),
    ("Change Execution",     "Network device APIs (RESTCONF/NETCONF) + Git rollback",    "required"),
    ("Caching Layer",        "Redis for exact+semantic cache (Ch 40)",                   "important"),
    ("Observability Stack",  "Prometheus + Grafana + LLM-as-judge quality monitor",     "important"),
    ("Scaling Layer",        "Kubernetes HPA on queue depth + continuous batching",      "important"),
    ("Security",             "JWT scopes, rate limiting, audit log for all AI actions",  "required"),
    ("Feedback Loop",        "Rejected recommendations -> RAG KB update -> retraining",  "important"),
    ("Runbook Updater",      "Docs Agent writes resolution back to KB after each fix",   "recommended"),
]

print(f"{'Component':<28} {'Production Equivalent':<45} {'Priority'}")
print("-" * 85)
for component, prod, priority in checklist:
    icon = "!!" if priority == "required" else (" *" if priority == "important" else "  ")
    print(f"{icon} {component:<26} {prod:<45} {priority}")

print()
print("Key lessons from production deployments:")
lessons = [
    "Start with Crawl phase - never jump to Run autonomy from day 1",
    "Always show reasoning chain (finding + confidence + evidence), never conclusion alone",
    "High-risk actions (core config changes) ALWAYS require human approval - no exceptions",
    "Track recommendation diversity - alert if AI recommends same fix type >40% of the time",
    "DLQ is not a failure - it is an isolation mechanism for problematic incidents",
    "The 0-to-60, 60-to-100 problem: last 20% of edge cases = 80% of engineering effort",
    "Degenerate feedback loops: track whether AI is reasoning or just pattern-matching",
]
for i, lesson in enumerate(lessons, 1):
    print(f"  {i}. {lesson}")

print()
print("=" * 65)
print(f"Total LLM API calls in this notebook: {CALL_COUNT[0]}")
print(f"Estimated notebook cost:               ${CALL_COUNT[0] * 0.0015:.4f}")
print("=" * 65)
print()
print("You have now built a complete NetOps AI platform from scratch.")
print("Every component, every layer, every failure mode — understood and implemented.")
print()
print("Volume 4: Federated AI across networks. Stay tuned!")